<a href="https://colab.research.google.com/github/Lu120706/Analitics-/blob/main/Analictics_gyj.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install Flask

In [5]:
from flask import Flask, render_template_string, request, redirect, url_for
from datetime import datetime
import pytz


In [35]:
usuarios = {
    "admin": {"password": "123", "empresa": "Organizacion GyJ" },
    "juan": {"password": "123", "empresa": "Colmena"},
    "maria": {"password": "123", "empresa": "Almasa"},
    "mario": {"password": "123", "empresa": "GyJ Ferreterias"}
}

contactos_tic = [
    {"nombre": "Jose Pérez", "rol": "Jefe TIC", "email": "jose.perez@empresa.com"},
    {"nombre": "Juliana Gómez", "rol": "Soporte TIC", "email": "juliana.gomez@empresa.com"},
    {"nombre": "Carlos López", "rol": "Analista TIC", "email": "carlos.lopez@empresa.com"}
]

empresas_config = {
    "Colmena": {
        "logos": ["https://cdn.phototourl.com/member/2026-03-24-8c607f2a-fbbf-45ae-a635-2b91d8928fae.png"],
        "pbi": "URL_PBI_COLMENA",
        "noticias": [{"titulo": "Aviso Colmena", "texto": "Hoy tenemos mantenimiento..."}]
    },
    "Almasa": {
        "logos": ["https://cdn.phototourl.com/member/2026-03-24-3b7755ae-b4ae-4423-b0a9-a70e72c6a815.png"],
        "pbi": "https://app.powerbi.com/groups/7142c5d0-c2b4-4821-a8e5-3a4c3c5deb23/list?tenant=87e3faef-89ab-4f30-9957-56b5f90aad08&experience=power-bi",
        "noticias": [{"titulo": "Aviso Almasa", "texto": "Nuevas políticas de seguridad."}]
    },
    "GyJ Ferreterias": {
        "logos": ["https://cdn.phototourl.com/member/2026-03-24-cd01a241-a822-499e-880f-81e6e291a732.png"],
        "pbi": "URL_PBI_GYJ",
        "noticias": [{"titulo": "Aviso GyJ", "texto": "Cierre de inventario este fin de semana."}]
    },
    "todas": {
        "logos": [
            "https://cdn.phototourl.com/member/2026-03-24-cd01a241-a822-499e-880f-81e6e291a732.png",
            "https://cdn.phototourl.com/member/2026-03-24-8c607f2a-fbbf-45ae-a635-2b91d8928fae.png",
            "https://cdn.phototourl.com/member/2026-03-24-3b7755ae-b4ae-4423-b0a9-a70e72c6a815.png"
        ],
        "pbi": "URL_PBI_GLOBAL",
        "noticias": [{"titulo": "Global", "texto": "Comunicado general para la organización."}]
    }
}
login_html = """
<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <title>Login</title>
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
    <style>
        body {
            background-color: #000;
            height: 100vh;
            color: white;
            display: flex;
            justify-content: center;
            align-items: center;
        }
        .login-box {
            background: #111;
            padding: 30px;
            border-radius: 12px;
            width: 100%;
            max-width: 400px;
        }
        .login-box h2 {
            margin-bottom: 20px;
        }
    </style>
</head>
<body>
    <div class="login-box text-center">
        <h2>Iniciar Sesión</h2>
        {% if error %}
            <div class="alert alert-danger">{{ error }}</div>
        {% endif %}
        <form method="post">
            <div class="mb-3 text-start">
                <label for="usuario" class="form-label">Usuario</label>
                <input type="text" class="form-control" name="usuario" required>
            </div>
            <div class="mb-3 text-start">
                <label for="password" class="form-label">Contraseña</label>
                <input type="password" class="form-control" name="password" required>
            </div>
            <button type="submit" class="btn btn-warning w-100">Entrar</button>
        </form>
    </div>
</body>
</html>
"""

# Dashboard HTML
dashboard_html = """
<!DOCTYPE html>
<html lang="es" data-bs-theme="dark">
<head>
    <meta charset="UTF-8">
    <title>Panel de Control</title>
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">

    <style>
        :root { --color-text: #ffffff; }

        body {
            background:
                linear-gradient(rgba(0,0,0,0.65), rgba(0,0,0,0.75)),
                url("https://cdn.phototourl.com/free/2026-03-24-f64f6bd2-15ea-4864-bb6a-f22ac718e417.jpg");
            background-size: cover;
            background-position: center;
            background-attachment: fixed;
            color: var(--color-text);
        }

        .navbar-custom { background-color: #F2E124 !important; box-shadow: 0 2px 6px rgba(0,0,0,0.4); }
        .navbar-custom .nav-link, .navbar-custom .navbar-brand { color: #000 !important; }
        .navbar-custom .nav-link::after {
            content: ""; position: absolute; width: 0%; height: 2px; bottom: 0; left: 0; background-color: #000; transition: 0.3s;
        }
        .navbar-custom .nav-link:hover::after { width: 100%; }

        .card { background-color: transparent; color: #fff; }

        /* Sidebar derecho para logos */
        .col-md-3 { border-left: 1px solid rgba(255,255,255,0.1); padding-left: 0; display: flex; justify-content: flex-end; }
        .logos-container { display: flex; flex-direction: column; gap: 0.25rem; align-items: flex-end; }

        .card.logo-card {
            width: 120px;
            height: 120px;
            display: flex;
            justify-content: center;
            align-items: center;
        }
        .card.logo-card img {
            max-width: 100%;
            max-height: 100%;
            object-fit: contain;
        }
        .card.logo-card:hover { transform: scale(1.05); }

        h1, h3, p { text-shadow: 0 2px 8px rgba(0,0,0,0.8); }

          .powerbi-container {
              margin-top: 40px;
              width: 100%;
              max-width: 800px; /*  controla el ancho */
              margin-left: auto;
              margin-right: auto;
          }

          .powerbi-container iframe {
              width: 100%;
              height: 800px; /* controla la altura */
              border: 0;
          }
    </style>
</head>
<body>

<nav class="navbar navbar-expand-lg navbar-custom px-4">
    <div class="container-fluid">
        <a class="navbar-brand fw-bold" href="#">{{ empresa.upper() }}</a>
        <button class="navbar-toggler" type="button" data-bs-toggle="collapse" data-bs-target="#navbarNav">
            <span class="navbar-toggler-icon"></span>
        </button>
        <div class="collapse navbar-collapse" id="navbarNav">
            <ul class="navbar-nav ms-auto">
                <li class="nav-item"><a class="nav-link fw-semibold" href="/mercadeo?usuario={{usuario}}&empresa={{empresa}}">Mercadeo</a></li>
                <li class="nav-item"><a class="nav-link fw-semibold" href="/talento?usuario={{usuario}}&empresa={{empresa}}">Talento Humano</a></li>
                <li class="nav-item"><a class="nav-link fw-semibold" href="/gerencia?usuario={{usuario}}&empresa={{empresa}}">Gerencia</a></li>
            </ul>
            <div class="dropdown ms-3 d-flex align-items-center">
                <button class="btn btn-outline-dark dropdown-toggle rounded-pill px-4" type="button" data-bs-toggle="dropdown">
                    👤 {{ usuario.upper() }}
                </button>
                <ul class="dropdown-menu dropdown-menu-end shadow border-0">
                    <li><span class="dropdown-item-text">Empresa: <strong>{{ empresa }}</strong></span></li>
                    <li><hr class="dropdown-divider"></li>
                    <li><a class="dropdown-item text-danger fw-bold" href="/">Cerrar sesión</a></li>
                </ul>
            </div>
        </div>
    </div>
</nav>

<div class="container-fluid mt-4">
    <div class="row align-items-start">

        <!-- Columna principal -->
        <div class="col-md-9 ps-md-5">
            <h1 class="display-6 fw-bold text-center">{{ saludo }}, {{ usuario }}</h1>

            <!-- NOTICIAS -->
            <div class="mt-4 text-start mx-auto" style="max-width: 900px;">
                <h3 class="mb-4 border-bottom pb-2">Noticias Destacadas</h3>

                {% for noticia in noticias %}
                    <div class="mb-3 p-3 rounded" style="background-color: rgba(0,0,0,0.4);">
                        <h5>{{ noticia.titulo }}</h5>
                        <p>{{ noticia.texto }}</p>
                    </div>
                {% endfor %}
            </div>

          <!-- INFORME -->
          <div class="powerbi-container" style="position: relative; left: -40px;">
              <h3 class="mt-5 mb-3 border-bottom pb-2">Informe OR</h3>
              <iframe
                  title="Informe OR"
                  src="https://app.powerbi.com/view?r=eyJrIjoiYmQwNDljYzEtY2EwMS00YWU1LWIyYzctODBiZGMxMmE2YWE5IiwidCI6Ijg3ZTNmYWVmLTg5YWItNGYzMC05OTU3LTU2YjVmOTBhYWQwOCJ9"
                  style="width: 100%; height: 700px; border: none; border-radius: 10px;">
              </iframe>
          </div>
        </div>

        <!--LOGOS -->
        <div class="col-md-3 d-flex flex-column align-items-end mt-4">
            {% for url in logos %}
                <div class="card logo-card text-center mb-3" style="width:120px; height:120px;">
                    <img src="{{ url }}" style="width:100%; height:100%; object-fit:contain;">
                </div>
            {% endfor %}
        </div>

        <!-- CONTACTOS TIC -->
        <div class="col-md-5 offset-md-1 d-flex flex-column align-items-start mt-5">
            <h3 class="mb-4 border-bottom pb-2">Contactos TIC</h3>

            <div class="row g-4">
                {% for contacto in contactos_tic %}
                <div class="col-md-4">
                    <div class="p-5 rounded h-200 d-flex flex-column justify-content-between"
                        style="background-color: rgba(0,0,0,0.4); min-height: 300px;">

                        <div>
                            <h5 class="mb-2">{{ contacto.nombre }}</h5>
                            <p class="mb-2 text-muted">{{ contacto.rol }}</p>
                        </div>

                        <div>
                            <p class="mb-2">
                                Email: <a href="mailto:{{ contacto.email }}" style="color:#F2E124;">
                                    {{ contacto.email }}
                                </a>
                            </p>

                            {% if contacto.telefono %}
                            <p class="mb-0">Tel: {{ contacto.telefono }}</p>
                            {% endif %}
                        </div>

                    </div>
                </div>
                {% endfor %}
            </div>
        </div>
    </div>
</div>

<script src="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/js/bootstrap.bundle.min.js"></script>
</body>
</html>
"""

gerencia_html = """
<!DOCTYPE html>
<html lang="es" data-bs-theme="dark">
<head>
    <meta charset="UTF-8">
    <title>Gerencia</title>
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">

    <style>
        :root { --color-text: #ffffff; }

        body {
            background:
                linear-gradient(rgba(0,0,0,0.65), rgba(0,0,0,0.75)),
                url("https://cdn.phototourl.com/free/2026-03-24-f64f6bd2-15ea-4864-bb6a-f22ac718e417.jpg");
            background-size: cover;
            background-position: center;
            background-attachment: fixed;
            color: var(--color-text);
        }

        .navbar-custom { background-color: #F2E124 !important; }
        .navbar-custom .nav-link, .navbar-custom .navbar-brand { color: #000 !important; }

        .col-md-3 { display: flex; justify-content: flex-end; }
        .logos-container { display: flex; flex-direction: column; gap: 0.25rem; align-items: flex-end; }

        .card.logo-card {
            width: 120px;
            height: 120px;
            display: flex;
            justify-content: center;
            align-items: center;
        }

        .card.logo-card img {
            max-width: 100%;
            max-height: 100%;
            object-fit: contain;
        }
    </style>
</head>

<body>

<nav class="navbar navbar-expand-lg navbar-custom px-4">
    <div class="container-fluid">
        <a class="navbar-brand fw-bold" href="/dashboard?usuario={{usuario}}&empresa={{empresa}}">{{ empresa.upper() }}</a>
      <div class="navbar-nav ms-auto">
      <a class="nav-link" href="/gerencia?usuario={{usuario}}&empresa={{empresa}}&seccion=indicadores">Indicadores</a>
      <a class="nav-link" href="/gerencia?usuario={{usuario}}&empresa={{empresa}}&seccion=proyecciones">Proyecciones</a>
      <a class="nav-link" href="/gerencia?usuario={{usuario}}&empresa={{empresa}}&seccion=reportes">Reportes</a>
      </div>
    </div>
</nav>

<div class="container-fluid mt-4">
    <div class="row">

        <!-- CONTENIDO -->
        <div class="col-md-9">
            <h1 class="text-center">Gerencia</h1>

    {% if seccion == "Indicadores" %}

        <h3 class="mt-4 border-bottom pb-2">Indicadores</h3>

        <div class="mb-3 p-3 rounded" style="background-color: rgba(0,0,0,0.4);">
            <p>Indicadores</p>
        </div>

    {% elif seccion == "Proyecciones" %}

        <h3 class="mt-4 border-bottom pb-2">Proyecciones</h3>

        <div class="mb-3 p-3 rounded" style="background-color: rgba(0,0,0,0.4);">
            <p>Metas del mes</p>
        </div>

    {% elif seccion == "Reportes" %}

        <h3 class="mt-4 border-bottom pb-2"> Reportes</h3>

        <div class="mb-3 p-3 rounded" style="background-color: rgba(0,0,0,0.4);">
            <p>Reportes.</p>
        </div>

    {% else %}

        <h3 class="mt-4 border-bottom pb-2">Gestión Comercial</h3>

        <div class="mb-3 p-3 rounded" style="background-color: rgba(0,0,0,0.4);">
            <h5>Campañas</h5>
            <p>Seguimiento a campañas activas y resultados.</p>
        </div>

        <div class="mb-3 p-3 rounded" style="background-color: rgba(0,0,0,0.4);">
            <h5>Clientes</h5>
            <p>Segmentación y comportamiento de clientes.</p>
        </div>

        <div class="mb-3 p-3 rounded" style="background-color: rgba(0,0,0,0.4);">
            <h5>Ventas</h5>
            <p>Análisis de desempeño comercial.</p>
        </div>

    {% endif %}
</div>

        <!-- LOGOS -->
        <div class="col-md-3 d-flex flex-column align-items-end mt-4">
            {% for url in logos %}
                <div class="card logo-card mb-3">
                    <img src="{{ url }}">
                </div>
            {% endfor %}
        </div>

      </div>
  </div>

  </body>
  </html>
  """

talento_html = """
<!DOCTYPE html>
<html lang="es" data-bs-theme="dark">
<head>
    <meta charset="UTF-8">
    <title>Talento Humano</title>
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">

    <style>
        :root { --color-text: #ffffff; }

        body {
            background:
                linear-gradient(rgba(0,0,0,0.65), rgba(0,0,0,0.75)),
                url("https://cdn.phototourl.com/free/2026-03-24-f64f6bd2-15ea-4864-bb6a-f22ac718e417.jpg");
            background-size: cover;
            background-position: center;
            background-attachment: fixed;
            color: var(--color-text);
        }

        .navbar-custom { background-color: #F2E124 !important; }
        .navbar-custom .nav-link, .navbar-custom .navbar-brand { color: #000 !important; }

        .col-md-3 { display: flex; justify-content: flex-end; }

        .card.logo-card {
            width: 120px;
            height: 120px;
            display: flex;
            justify-content: center;
            align-items: center;
        }

        .card.logo-card img {
            max-width: 100%;
            max-height: 100%;
            object-fit: contain;
        }
    </style>
</head>

<body>

<nav class="navbar navbar-expand-lg navbar-custom px-4">
    <div class="container-fluid">
        <a class="navbar-brand fw-bold" href="/dashboard?usuario={{usuario}}&empresa={{empresa}}">
            {{ empresa.upper() }}
        </a>

        <div class="navbar-nav ms-auto">
            <a class="nav-link" href="/talento?usuario={{usuario}}&empresa={{empresa}}&seccion=Empleados">Empleados</a>
            <a class="nav-link" href="/talento?usuario={{usuario}}&empresa={{empresa}}&seccion=Capacitaciones">Capacitaciones</a>
            <a class="nav-link" href="/talento?usuario={{usuario}}&empresa={{empresa}}&seccion=Ventas">Ventas</a>
        </div>
    </div>
</nav>

<div class="container-fluid mt-4">
    <div class="row">

        <!-- CONTENIDO -->
        <div class="col-md-9">
            <h1 class="text-center">Talento Humano</h1>

            {% if seccion == "Empleados" %}

                <h3 class="mt-4 border-bottom pb-2">Empleados</h3>

                <div class="mb-3 p-3 rounded" style="background-color: rgba(0,0,0,0.4);">
                    <p>Seguimiento de empleados y demás.</p>
                </div>

            {% elif seccion == "Capacitaciones" %}

                <h3 class="mt-4 border-bottom pb-2">Capacitaciones</h3>

                <div class="mb-3 p-3 rounded" style="background-color: rgba(0,0,0,0.4);">
                    <p>Capacitaciones de nuevo personal.</p>
                </div>

            {% elif seccion == "Ventas" %}

                <h3 class="mt-4 border-bottom pb-2">Ventas</h3>

                <div class="mb-3 p-3 rounded" style="background-color: rgba(0,0,0,0.4);">
                    <p>Análisis de desempeño comercial.</p>
                </div>

            {% else %}

                <h3 class="mt-4 border-bottom pb-2">Resumen General</h3>

                <div class="mb-3 p-3 rounded" style="background-color: rgba(0,0,0,0.4);">
                    <h5>Empleados</h5>
                    <p>Gestión del personal.</p>
                </div>

                <div class="mb-3 p-3 rounded" style="background-color: rgba(0,0,0,0.4);">
                    <h5>Capacitaciones</h5>
                    <p>Formación del equipo.</p>
                </div>

                <div class="mb-3 p-3 rounded" style="background-color: rgba(0,0,0,0.4);">
                    <h5>Ventas</h5>
                    <p>Relación con resultados comerciales.</p>
                </div>

            {% endif %}
        </div>

        <!-- LOGOS -->
        <div class="col-md-3 d-flex flex-column align-items-end mt-4">
            {% for url in logos %}
                <div class="card logo-card mb-3">
                    <img src="{{ url }}">
                </div>
            {% endfor %}
        </div>

    </div>
</div>

</body>
</html>
"""

mercadeo_html = """
<!DOCTYPE html>
<html lang="es" data-bs-theme="dark">
<head>
    <meta charset="UTF-8">
    <title>Mercadeo</title>
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">

    <style>
        :root { --color-text: #ffffff; }

        body {
            background:
                linear-gradient(rgba(0,0,0,0.65), rgba(0,0,0,0.75)),
                url("https://cdn.phototourl.com/free/2026-03-24-f64f6bd2-15ea-4864-bb6a-f22ac718e417.jpg");
            background-size: cover;
            background-position: center;
            background-attachment: fixed;
            color: var(--color-text);
        }

        .navbar-custom { background-color: #F2E124 !important; }
        .navbar-custom .nav-link, .navbar-custom .navbar-brand { color: #000 !important; }

        .col-md-3 { display: flex; justify-content: flex-end; }

        .card.logo-card {
            width: 120px;
            height: 120px;
            display: flex;
            justify-content: center;
            align-items: center;
        }

        .card.logo-card img {
            max-width: 100%;
            max-height: 100%;
            object-fit: contain;
        }
    </style>
</head>

<body>

<nav class="navbar navbar-expand-lg navbar-custom px-4">
    <div class="container-fluid">
        <a class="navbar-brand fw-bold" href="/dashboard?usuario={{usuario}}&empresa={{empresa}}">
            {{ empresa.upper() }}
        </a>

        <div class="navbar-nav ms-auto">
        <a class="nav-link" href="/mercadeo?usuario={{usuario}}&empresa={{empresa}}&seccion=campanas">Campañas</a>
        <a class="nav-link" href="/mercadeo?usuario={{usuario}}&empresa={{empresa}}&seccion=clientes">Clientes</a>
        <a class="nav-link" href="/mercadeo?usuario={{usuario}}&empresa={{empresa}}&seccion=ventas">Ventas</a>
        </div>
    </div>
</nav>

<div class="container-fluid mt-4">
    <div class="row">

        <!-- CONTENIDO -->
<div class="col-md-9">
    <h1 class="text-center">Mercadeo</h1>

    {% if seccion == "campanas" %}

        <h3 class="mt-4 border-bottom pb-2">Campañas</h3>

        <div class="mb-3 p-3 rounded" style="background-color: rgba(0,0,0,0.4);">
            <p>Seguimiento a campañas activas y resultados.</p>
        </div>

    {% elif seccion == "clientes" %}

        <h3 class="mt-4 border-bottom pb-2">Clientes</h3>

        <div class="mb-3 p-3 rounded" style="background-color: rgba(0,0,0,0.4);">
            <p>Segmentación y comportamiento de clientes.</p>
        </div>

    {% elif seccion == "ventas" %}

        <h3 class="mt-4 border-bottom pb-2"> Ventas</h3>

        <div class="mb-3 p-3 rounded" style="background-color: rgba(0,0,0,0.4);">
            <p>Analisis de desempeño comercial.</p>
        </div>

    {% else %}

        <h3 class="mt-4 border-bottom pb-2">Gestión Comercial</h3>

        <div class="mb-3 p-3 rounded" style="background-color: rgba(0,0,0,0.4);">
            <h5>Campañas</h5>
            <p>Seguimiento a campañas activas y resultados.</p>
        </div>

        <div class="mb-3 p-3 rounded" style="background-color: rgba(0,0,0,0.4);">
            <h5>Clientes</h5>
            <p>Segmentación y comportamiento de clientes.</p>
        </div>

        <div class="mb-3 p-3 rounded" style="background-color: rgba(0,0,0,0.4);">
            <h5>Ventas</h5>
            <p>Análisis de desempeño comercial.</p>
        </div>

    {% endif %}
</div>

        <!-- LOGOS -->
        <div class="col-md-3 d-flex flex-column align-items-end mt-4">
            {% for url in logos %}
                <div class="card logo-card mb-3">
                    <img src="{{ url }}">
                </div>
            {% endfor %}
        </div>

    </div>
</div>

</body>
</html>
"""

app = Flask(__name__)

if not hasattr(app, "_has_registered_routes"):
    # LOGIN
    @app.route("/", methods=["GET", "POST"])
    def login_route():
        error = ""
        if request.method == "POST":
            username = request.form.get("usuario")
            password = request.form.get("password")
            if username in usuarios and usuarios[username]["password"] == password:
                empresa_auto = usuarios[username]["empresa"]
                return redirect(url_for("dashboard_route", usuario=username, empresa=empresa_auto))
            error = "Credenciales inválidas"
        return render_template_string(login_html, error=error)


    # DASHBOARD
    @app.route("/dashboard")
    def dashboard_route():
        usuario = request.args.get("usuario", "Usuario")
        empresa = request.args.get("empresa", "Empresa")

        config = empresas_config.get(empresa, empresas_config["todas"])
        logos = config["logos"]

        noticias = [
            {"titulo": "Nueva actualización", "texto": "El sistema fue mejorado."},
            {"titulo": "Recordatorio", "texto": "Revisar reportes pendientes."}
        ]

        zona_colombia = pytz.timezone("America/Bogota")
        hora = datetime.now(zona_colombia).hour

        if hora < 12:
            saludo = "Buenos días"
        elif hora < 18:
            saludo = "Buenas tardes"
        else:
            saludo = "Buenas noches"

        return render_template_string(
            dashboard_html,
            usuario=usuario,
            empresa=empresa,
            noticias=noticias,
            logos=logos,
            saludo=saludo,
            contactos_tic=contactos_tic
        )


    @app.route("/mercadeo")
    def mercadeo_modulo():
        usuario = request.args.get("usuario", "Usuario")
        empresa = request.args.get("empresa", "Empresa")
        seccion = request.args.get("seccion", "inicio")

        config = empresas_config.get(empresa, empresas_config["todas"])
        logos = config["logos"]

        return render_template_string(
            mercadeo_html,
            usuario=usuario,
            empresa=empresa,
            logos=logos,
            seccion=seccion
        )

    @app.route("/talento")
    def talento_modulo():
        usuario = request.args.get("usuario", "Usuario")
        empresa = request.args.get("empresa", "Empresa")
        seccion = request.args.get("seccion", "inicio")

        config = empresas_config.get(empresa, empresas_config["todas"])
        logos = config["logos"]

        return render_template_string(
            talento_html,
            usuario=usuario,
            empresa=empresa,
            logos=logos,
            seccion=seccion
        )


    @app.route("/gerencia")
    def gerencia_modulo():
        usuario = request.args.get("usuario", "Usuario")
        empresa = request.args.get("empresa", "Empresa")
        seccion = request.args.get("seccion", "inicio")

        config = empresas_config.get(empresa, empresas_config["todas"])
        logos = config["logos"]

        return render_template_string(
            gerencia_html,
            usuario=usuario,
            empresa=empresa,
            logos=logos,
            seccion=seccion
        )

    app._has_registered_routes = True

In [36]:
from google.colab import output

if __name__ == "__main__":
    output.serve_kernel_port_as_window(5000)
    app.run(host="0.0.0.0", port=5000)


Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [30/Mar/2026 19:56:13] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [30/Mar/2026 19:56:16] "POST / HTTP/1.1" 302 -
INFO:werkzeug:127.0.0.1 - - [30/Mar/2026 19:56:17] "GET /dashboard?usuario=admin&empresa=Organizacion+GyJ HTTP/1.1" 200 -
